# Road Network Extraction and Processing

This notebook handles the full pipeline for extracting, cleaning, and characterizing urban road networks from OpenStreetMap (OSM). It produces the processed graph files and summary statistics used in downstream analyses.

Documentation Note: Portions of this documentation were drafted with the assistance of Claude (Anthropic) and subsequently reviewed, edited, and verified for technical accuracy by the authors.

---
## Section 1: Import Libraries

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import sys
import pickle
import re
import random

# ── Progress bars ─────────────────────────────────────────────────────────────
from tqdm import tqdm
import shutil

# ── Geospatial ────────────────────────────────────────────────────────────────
import osmnx as ox
import networkx as nx
import geopandas as gpd
import shapely
from shapely.geometry import Point, Polygon, box, LineString

# ── Visualisation ─────────────────────────────────────────────────────────────
from matplotlib import pyplot as plt

# ── Numerical / Tabular ───────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Display configuration ─────────────────────────────────────────────────────
import warnings; warnings.simplefilter('ignore')
pd.options.display.max_columns = None
pd.options.display.max_rows = 30
np.set_printoptions(threshold=sys.maxsize)

---
## Section 2: Extract Road Networks from OpenStreetMap

### Context
Urban road networks are extracted for a global sample of cities using pre-computed convex hull boundary shapefiles. Each boundary defines the spatial extent of one urban area; the OSM API is queried to retrieve all driveable road segments (including service roads) within that polygon.

### Network type
The `drive_service` network type in OSMnx includes all edges accessible by motor vehicles, plus service roads (e.g., parking aisles, alleys). This is a superset of `drive`, chosen to capture informal road infrastructure common in lower-income urban contexts.

### Coordinate Reference System
- **EPSG:4326** — WGS84 geographic coordinates (longitude/latitude). Used here because OSMnx requires geographic input polygons. All spatial comparisons requiring metric distances are reprojected downstream.

### Inputs
- `shapefile_bnd_dir`: Directory of `.shp` files, one per settlement cluster. Each shapefile contains a single convex hull polygon and a network ID (`Id` field).

### Outputs
- One pickled `networkx.MultiDiGraph` per settlement cluster, saved to `raw_osmgraphs_drive_service_conv/`. Protocol 2 ensures compatibility with Python 2 environments if needed.

### Error handling
OSMnx raises exceptions for settlement clusters with no driveable road coverage in the queried polygon (e.g., uninhabited or water-covered areas). These are caught silently and the filename is printed for manual review.

In [ ]:
# ── Coordinate Reference System ───────────────────────────────────────────────
# EPSG:4326 = WGS84 geographic CRS (degrees lon/lat); required by OSMnx
crs_lonlat = {'init': 'epsg:' + str(4326)}

# ── Input: city boundary shapefiles ──────────────────────────────────────────
# Each .shp file contains one convex hull polygon bounding an settlement cluster,
# with an 'Id' attribute used as the unique network identifier.
shapefile_bnd_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology(2)/Research/Global road network resilience/01_data/4229_bnd_convex'

# ── Extract OSM road network for each city boundary ───────────────────────────
counter = 0  # Tracks number of successfully extracted networks

for i, file in enumerate(os.listdir(shapefile_bnd_dir)):
    if file[-4:] == '.shp':
        # Load boundary and reproject to WGS84
        bnd = gpd.read_file(shapefile_bnd_dir+file).to_crs(crs_lonlat)
        net_id = bnd['Id'][0]

        try:
            # Query OSM for the drive+service road network within the polygon.
            # Returns a MultiDiGraph (directed, allows parallel edges) with
            # nodes as intersections and edges as road segments.                 
            G = ox.graph_from_polygon(bnd.geometry[0], network_type='drive_service')
            counter += 1

            # Serialize graph to disk using pickle protocol 2 (Python 2 compatible)
            with open('/05_WB/CityResilience/1_processed/raw_osmgraphs_drive_service_conv/g_raw_drive_conv_' + str(net_id) + '.pk', 'wb') as handle:
                pickle.dump(G, handle, protocol=2)
        except:
            # Print the filename for manual inspection; common causes are
            # empty polygons, network coverage gaps, or OSM API timeouts.
            print(file)

---
## Section 3: Initial Graph Property Summary

### Purpose
Before any filtering or cleaning, this step computes a baseline characterization of each raw network. The summary is used to (a) understand the scale of the dataset, (b) check data quality, and (c) provide inputs for downstream filtering decisions.

### Metrics computed
| Column | Description |
|--------|-------------|
| `net_id` | Unique network identifier |
| `nodes` | Number of intersections/endpoints |
| `edges` | Number of road segments |
| `tot_length` | Total road length (meters), summed over all edge weights |
| `tertiary_length` | Combined length of `tertiary` + `tertiary_link` edges |
| `residential_length` | Length of `residential` edges |

### Note on `highway` attribute
The OSM `highway` tag classifies each road segment. In OSMnx graphs, this attribute is stored as a string or a list of strings (when a segment is tagged with multiple classes). This cell handles only scalar string values; multi-type edges are processed more carefully in later sections.

In [ ]:
# ── Initialise summary DataFrame ──────────────────────────────────────────────
# Pre-allocate for 4,194 networks (the expected count after initial extraction)
df = pd.DataFrame(columns=['net_id', 'nodes', 'edges','tot_length','tertiary_length','residential_length'], index=range(4194))
counter = 0

# ── Iterate over all saved raw graph files ────────────────────────────────────
# Calculate the lengths for tertiary as well as residential road types
for file in os.listdir(network_dir):

    # Load pickled NetworkX graph
    G = pickle.load(open(network_dir + file, 'rb'))

    # Accumulators for road-type-specific lengths
    tertiary_length = []
    residential_length = []

    # Extract network ID from filename (characters 6 onward, dropping the '.pk' extension)
    net_id = file[6:][:-3]

    # ── Graph-level statistics ────────────────────────────────────────────────
    # Populate dataframe with network id, number of nodes, number of edges, total road length
    df.at[counter, 'net_id'] = net_id
    df.at[counter, 'nodes'] = G.number_of_nodes()
    df.at[counter, 'edges'] = G.number_of_edges()
    df.at[counter, 'tot_length'] = G.size(weight="length")


    # ── Edge-level road-type classification ───────────────────────────────────
    # Iterate over all directed edges; each edge carries a 'highway' attribute.
    # 'tertiary_link' edges connect tertiary roads to other road classes and
    # are grouped with tertiary for length accounting.

    # Examine the "highway" attribute of the OSM networks/graphs for road lengths
    for i,j,data in G.edges.data():
        if data['highway'] == 'tertiary':
            tertiary_length.append(data['length'])
        elif data['highway'] == 'tertiary_link':
            tertiary_length.append(data['length'])
        elif data['highway'] == 'residential':
            residential_length.append(data['length'])
    
    # ── Store aggregated lengths ──────────────────────────────────────────────
    # Populate dataframe with tertiary road length and residential road length
    df.at[counter, 'tertiary_length'] = sum(tertiary_length)
    df.at[counter, 'residential_length'] = sum(residential_length)
    counter += 1

# ── Export ────────────────────────────────────────────────────────────────────
df.to_csv('/05_WB/CityResilience/1_processed/graphs_drive_service_tertiary_road_lengths_4194.csv')

---
## Section 4: Audit OSM Highway Classifications

### Purpose
OSM `highway` tags are assigned by volunteer contributors and vary in consistency across regions. This audit enumerates all unique highway classes present in each network and aggregates their frequency across the full dataset. The output informs decisions about which road types to retain, filter, or collapse in later processing steps.

### Handling nested lists
In OSMnx, edges with multiple overlapping OSM ways can inherit a *list* of highway tags instead of a single string (e.g., `['residential', 'service']`). The helper function `removeNestings` recursively flattens these nested structures into a single flat list before computing the unique set of types per network.

### Output
- Per-network DataFrame: each row stores a network ID and its set of highway classes.
- Aggregated count table: how many networks contain each highway class.

In [ ]:
# Function to remove list nestings
def reemovNestings(l):
    """
    Recursively flatten an arbitrarily nested list into a single flat list.

    OSMnx edge attributes can store highway tags as either a plain string or
    a list of strings (when a segment matches multiple OSM ways). This function
    handles both cases uniformly.

    Parameters
    ----------
    nested : iterable
        The (potentially nested) collection to flatten. May contain strings
        or further nested lists.
    flat_list : list
        Output accumulator. Pass an empty list [] on initial call.

    Returns
    -------
    None
        Results are appended in-place to `flat_list`.
    """
    for i in l: 
        if type(i) == list: 
            reemovNestings(i)
        else: 
            output.append(i)


# ── Per-network highway class audit ───────────────────────────────────────────
# Initiate dataframe to store road types
df = pd.DataFrame(columns=['net_id', 'net_types'], index=range(4190))
counter = 0
net_type_lst = [] # Initiate list of network types

# For each network, summarize road type
for file in os.listdir(network_dir):
    G = pickle.load(open(network_dir + file, 'rb'))
    net_id = file[6:][:-3]

    df.at[counter, 'net_id'] = net_id

    # Retrieve all 'highway' edge attributes as a dict {(u,v,key): tag}
    highway_class = nx.get_edge_attributes(G,'highway')
    output = [] 
    reemovNestings(highway_class.values())
    df.at[counter, 'net_types'] = list(set(output))
    net_type_lst.append(list(set(output)))
    counter += 1
    
output = []
reemovNestings(net_type_lst)

road_type_count_df = pd.DataFrame(columns=['highway_class', 'count'], index=range(171))
counter = 0

for item in list(set(output)):
    road_type_count_df.at[counter, 'highway_class'] = item
    road_type_count_df.at[counter, 'count'] = output.count(item)
    counter += 1
    
# ── Export ────────────────────────────────────────────────────────────────────
road_type_count_df.to_csv('/05_WB/CityResilience/1_processed/Drive_service_conv_highway_class_count.csv')

---
## Section 5: Remove Residential Roads

### Procedure
1. Copy the graph (`G2 = G.copy()`) to avoid mutating the original.
2. Identify all edges where the `highway` attribute equals `'residential'` **or** contains the substring `'residential'` (to catch rare compound tags).
3. Remove those edges from the copy.
4. Remove **isolated nodes** — intersections left with no connected edges — since they carry no meaningful topological information after the edge removal.
5. Save the cleaned graph.

### Note on edge mutation during iteration
Edges are collected from the **original** graph `G` and removed from the **copy** `G2`. This avoids the `RuntimeError: dictionary changed size during iteration` that would occur if modifying the graph being iterated.

In [ ]:
# Initialize directory to store processed networks/graphs
network_dir = 'L:/yiyi/graphs_cov_no_res2/'

# ── Remove residential roads from each network ────────────────────────────────
# Remove residential roads
for file in tqdm(os.listdir(network_dir)):
    # Extract network ID from filename (15 chars of prefix, drop '.pk' extension)
    net_id = file[15:][:-3]

    # Load original graph
    G = pickle.load(open(network_dir + file, 'rb'))

    # Work on a copy to avoid modifying the original data
    G2 = G.copy()

    # Identify residential edges from the original graph
    # Two conditions:
    #   (a) exact match: 'residential'
    #   (b) substring match: catches compound tags like 'residential;service'
    for i,j,data in G.edges.data():
        if data['highway'] == 'residential':
            G2.remove_edge(i, j)
        elif  'residential' in data['highway']:
            G2.remove_edge(i, j)

    # Remove isolated nodes as a result of the edge removal
    G2.remove_nodes_from(list(nx.isolates(G2)))

    # Save cleaned graph
    with open('L:yiyi/graphs_cov_no_res3/graph_cov_no_r_' + str(net_id) + '.pk', 'wb') as handle:
                pickle.dump(G2, handle, protocol=2)

---
## Section 6: Validate Spatial Coverage via Convex Hull Comparison

### Purpose
OSMnx does not always return road segments covering the entire input polygon — coverage may be sparser in peri-urban zones or areas where OSM data is incomplete. This section quantifies the spatial coverage gap by comparing:

- **Original convex hull area**: the area of the input boundary shapefile polygon (ground truth).
- **Graph convex hull area**: the area of the convex hull computed from the actual node coordinates returned by OSM.

The `missing_percentage` metric measures what fraction of the intended study area lacks OSM road coverage. Networks with high missing percentages may be unreliable for resilience analysis and should be filtered.

### Knee-point analysis
A cumulative distribution of `missing_percentage` across all networks is constructed. The **knee point** (elbow) of this curve — detected with `KneeLocator` — identifies a natural threshold separating well-covered from poorly-covered networks, and guides the decision of how many networks to retain for analysis.

### CRS note
Areas are computed in **EPSG:6933** (WGS 84 / NSIDC EASE-Grid 2.0 Global), an equal-area projection, which correctly preserves area measurements across the globe. Geographic CRS (EPSG:4326) must *not* be used for area calculations.

In [ ]:
# ── Compute convex hull area from graph node positions ────────────────────────
graph_convex_hull_shp_dir = 'L:/yiyi/4190_graph_cov/'
graph_conv_sqkm_df = pd.DataFrame(columns=['net_id', 'graph_conv_sqkm'], index=range(4188))
counter = 0

for file in tqdm(os.listdir(graph_convex_hull_shp_dir)):
    if file[-3:] == 'shp':
        # Extract net_id from filename; skip the first 2 characters and last 14
        # (which correspond to filename prefix/suffix wrapping the numeric ID)
        net_id = int(file[2:][:-14])

        # Read convex hull shapefile and reproject to equal-area CRS for metric area
        data = gpd.read_file(graph_convex_hull_shp_dir+file)
        data_meters = data.to_crs({'init': 'epsg:6933'})

        # Convert area from m² to km²
        sqkm = data_meters['geometry'].area/ 10**6
        
        graph_conv_sqkm_df.at[counter, 'net_id'] = net_id
        graph_conv_sqkm_df.at[counter, 'graph_conv_sqkm'] = sqkm
        
        counter += 1

# Unwrap the GeoSeries to a scalar float for clean CSV export        
graph_conv_sqkm_df['graph_conv_sqkm_simplified'] = graph_conv_sqkm_df.apply(lambda row: float(row['graph_conv_sqkm']), axis=1)
graph_conv_sqkm_df[['net_id', 'graph_conv_sqkm_simplified']].to_csv('L:/yiyi/graph_4188_conv_sqkm.csv')

# ── Load original (input) convex hull areas ───────────────────────────────────
# The original boundaries were pre-computed and stored with a 'python_sqkm' attribute
original_convex_hulls = gpd.read_file('H:/05_WB/CityResilience/1_processed/final_bnd/final_bnd/urcls_4190_conv_prop.shp')
original_convex_hulls_meters = original_convex_hulls.to_crs({'init': 'epsg:6933'})

In [ ]:
# ── Compute missing coverage percentage for each network ──────────────────────
# Inner join on net_id to align original vs. graph convex hull areas
ori_graph_conv_merge = pd.merge(original_convex_hulls_meters[['net_id', 'python_sqkm']],
                                graph_conv_sqkm_df[['net_id', 'graph_conv_sqkm_simplified']],
                                left_on = 'net_id',
                                right_on = 'net_id',
                                how = 'inner')

# missing_percentage: proportion of the original boundary area NOT covered by the
# graph's convex hull, expressed as a percentage.
# A value of 0 means the graph covers the entire intended area;
# a value of 50 means the graph covers only half of it.
ori_graph_conv_merge['missing_percentage'] = ori_graph_conv_merge.apply(lambda row:
                                                                        (row['python_sqkm'] - row['graph_conv_sqkm_simplified'])*100/row['python_sqkm'],
                                                                        axis = 1)

# ── Build cumulative count table vs. missing-percentage threshold ──────────────
# For each threshold t (1–100%), count how many networks have missing_percentage < t.
# This produces a cumulative distribution used to identify the knee point.
num_graph_df = pd.DataFrame(columns=['missing_percentage', 'count_of_4188'])
for i in range(100):
    missing_percentage = i+1
    count = ori_graph_conv_merge[ori_graph_conv_merge['missing_percentage']<missing_percentage].shape[0]
    num_graph_df.at[i, 'missing_percentage'] = missing_percentage
    num_graph_df.at[i, 'count_of_4188'] = count

num_graph_df.to_csv('L:/yiyi/missing_percentage_count_of_4188.csv')

# ── Knee-point detection ──────────────────────────────────────────────────────
# The KneeLocator identifies the point of maximum curvature in the cumulative
# coverage curve. This threshold separates the majority of well-covered networks
# from those with large spatial gaps, and guides the sample size decision.
y = list(num_graph_df['count_of_4188'].values)
x = range(1, len(y)+1)

kn = KneeLocator(x, y, curve='concave', direction='increasing')
print(kn.knee)

---
## Section 7: Comprehensive Road-Type Length Summary

### Purpose
This section computes a full breakdown of road length by OSM highway classification for all 4,190 cleaned networks. The resulting table is used to characterise the composition of each settlement cluster's road network and to compute road-type-specific metrics in later analyses.

### OSM highway hierarchy covered
| Class | Description |
|-------|-------------|
| `motorway` / `motorway_link` | Controlled-access highways |
| `trunk` / `trunk_link` | Major non-motorway through-roads |
| `primary` / `primary_link` | Primary national/regional roads |
| `secondary` / `secondary_link` | Secondary regional roads |
| `tertiary` / `tertiary_link` | Tertiary local connecting roads |
| `unclassified` | Minor roads without a specific classification |
| `service` | Access roads, parking aisles, driveways |
| `living_street` | Shared pedestrian-vehicle spaces |
| `road` | Unspecified/placeholder class |
| `residential` | Residential streets (retained here for reference, filtered elsewhere) |

### Implementation note
Each edge is classified into exactly one bucket using a simple equality check on the `highway` attribute. Edges with list-valued `highway` (multi-tagged segments) are not split across buckets here; they fall through to the default case. A more exhaustive treatment would handle list-valued tags explicitly.

In [ ]:
# ── Initialise summary DataFrame ──────────────────────────────────────────────
df = pd.DataFrame(columns=['net_id', 'nodes', 'edges','tot_length',
                           'primary_length',
                           'secondary_length',
                           'tertiary_length', 
                           'motorway_length', 
                           'unclassified_length',
                           'service_length',
                           'trunk_length', 
                           'living_street_length',
                           'road_length',
                           'residential_length' ], index=range(4190))
counter = 0

# ── Iterate over cleaned network files ────────────────────────────────────────
for file in os.listdir(network_dir):
    G = pickle.load(open(network_dir + file, 'rb'))
    
    primary_length = []
    secondary_length = []
    tertiary_length = []
    motorway_length = []
    unclassified_length = []
    service_length = []
    trunk_length = []
    living_street_length = []
    road_length = []
    residential_length = []
    
    # Extract net_id from filename
    net_id = file[15:][:-3]

    # Graph-level statistics
    df.at[counter, 'net_id'] = net_id
    df.at[counter, 'nodes'] = G.number_of_nodes()
    df.at[counter, 'edges'] = G.number_of_edges()
    df.at[counter, 'tot_length'] = G.size(weight="length")
    for i,j,data in G.edges.data():
        if data['highway'] == 'primary' or data['highway'] == 'primary_link':
            primary_length.append(data['length'])
            
        elif data['highway'] == 'secondary' or data['highway'] == 'secondary_link' :
            secondary_length.append(data['length'])
            
        elif data['highway'] == 'tertiary' or data['highway'] == 'tertiary_link':
            tertiary_length.append(data['length'])
        
        elif data['highway'] == 'motorway' or data['highway'] == 'motorway_link':
            motorway_length.append(data['length'])
            
        elif data['highway'] == 'unclassified':
            unclassified_length.append(data['length'])
        
        elif data['highway'] == 'service':
            service_length.append(data['length'])
        
        elif data['highway'] == 'trunk' or data['highway'] == 'trunk_link':
            trunk_length.append(data['length'])
            
        elif data['highway'] == 'living_street':
            living_street_length.append(data['length'])
        
        elif data['highway'] == 'road':
            road_length.append(data['length'])
        
        elif data['highway'] == 'residential':
            residential_length.append(data['length'])
            
    df.at[counter, 'primary_length'] = sum(primary_length) 
    df.at[counter, 'secondary_length'] = sum(secondary_length)
    df.at[counter, 'tertiary_length'] = sum(tertiary_length)
    df.at[counter, 'motorway_length'] = sum(motorway_length)
    df.at[counter, 'unclassified_length'] = sum(unclassified_length)
    df.at[counter, 'service_length'] = sum(service_length)
    df.at[counter, 'trunk_length'] = sum(trunk_length)
    df.at[counter, 'living_street_length'] = sum(living_street_length)
    df.at[counter, 'road_length'] = sum(road_length)
    df.at[counter, 'residential_length'] = sum(residential_length)
    counter += 1

# ── Export ────────────────────────────────────────────────────────────────────    
df.to_csv('L:/yiyi/drive_service_no_res_4190_graph_properties.csv')

---
## Section 8: Variance of Edge Traversal Frequency

### Context
Each network has an associated edge-attribute CSV containing `dry_route_freq` — the frequency with which each edge is traversed by simulated routes under dry (non-flood) conditions. This frequency distribution characterises how evenly traffic load is distributed across the road network.

### Metric: Variance
Variance measures the spread of edge traversal frequencies around the mean. A high variance indicates that a small number of roads carry disproportionately high traffic — a sign of network vulnerability where those roads become critical bottlenecks.

### Inputs
- `edge_df_dir`: directory of per-network CSVs. Filename format encodes the network ID at position index 4 when split by `_`.
- Column: `dry_route_freq` (float) — route frequency per edge under non-flood conditions.

### Outputs
- `edge_freq_var.csv`: DataFrame with columns `net_id` and `edge_freq_var`.

In [ ]:
# ── Input: per-network edge attribute tables ───────────────────────────────────
edge_df_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology(2)/Research/Global road network resilience/01_data/edge_attri_df'

# ── Accumulators ──────────────────────────────────────────────────────────────
netids = []
variances = []

# ── Compute variance for each network ─────────────────────────────────────────
# Loop through the files
for file in tqdm(os.listdir(edge_df_dir)):
    if file.endswith("csv"):
        # Load edge attribute table; index_col=0 drops the default integer index
        net_df = pd.read_csv(os.path.join(edge_df_dir,file), index_col=0)

        # Network ID is encoded in the 5th underscore-delimited token of the filename
        net_id = int(file.split('_')[4]) # extract network id

        # Calculate the variance of edge frequency
        edge_var = net_df.dry_route_freq.var()

        # Append to lists
        netids.append(net_id)
        variances.append(edge_var)


# ── Assemble and export results ───────────────────────────────────────────────
# Create new DataFrame
edge_var_df = pd.DataFrame({
    'net_id': netids,
    'edge_freq_var': variances
})

# Save to csv file
edge_var_df.to_csv('/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology(2)/Research/Global road network resilience/01_data/csv/edge_freq_var.csv')

100%|██████████| 2615/2615 [00:12<00:00, 211.86it/s]


---
## Section 9: Gini Coefficient of Edge Traversal Frequency

### Context
Variance is sensitive to the scale of the frequency values and can be dominated by extreme outliers. The **Gini coefficient** provides a scale-invariant, bounded measure of inequality in the distribution of edge traversal frequencies. It ranges from 0 (perfectly equal load sharing) to 1 (all traffic concentrated on a single edge).

### Mathematical definition
For a sorted non-negative array $x_1 \leq x_2 \leq \cdots \leq x_n$ with cumulative sum $S$:

$$G = \frac{n + 1 - 2 \cdot \frac{\sum_{i=1}^{n} (\text{cumsum})_i}{S}}{n}$$

This is equivalent to twice the area between the Lorenz curve and the line of perfect equality.

### Implementation notes
- NaN values (edges with missing frequency data) are dropped before computation.
- An array of all-zeros returns $G = 0$ by convention (uniform distribution at zero).
- Negative values raise a `ValueError` since the Gini is defined only for non-negative quantities.

In [ ]:
def gini(x):
    """
    Compute the Gini coefficient of a 1-D array of non-negative values.

    The Gini coefficient measures inequality in a distribution:
    - G = 0: perfectly equal (all edges have identical traversal frequency)
    - G = 1: maximally unequal (all traffic on one edge)

    Parameters
    ----------
    x : array-like of float
        Non-negative values (e.g., edge traversal frequencies). May contain NaNs,
        which are silently removed.

    Returns
    -------
    float
        Gini coefficient in [0, 1], or np.nan if the input is empty after NaN removal.

    Raises
    ------
    ValueError
        If any non-NaN value is negative.
    """
    x = np.asarray(x, dtype=float)

    # Drop NaN entries (edges with missing routing data)
    x = x[~np.isnan(x)]

    if np.amin(x) < 0:
        raise ValueError("Gini is not defined for negative values")

    if x.size == 0:
        return np.nan # Cannot compute Gini on an empty array

    # If all values are zero, define Gini = 0
    if np.all(x == 0):
        return 0.0  # By convention: zero vector → perfectly equal at zero

    # Sort ascending (required for the cumulative sum formula)
    x = np.sort(x)

    n = x.size
    cumx = np.cumsum(x)

    # Standard Gini formula based on the cumulative sum representation of the Lorenz curve
    gini_coeff = (n + 1 - 2 * np.sum(cumx) / cumx[-1]) / n
    return gini_coeff

In [ ]:
# ── Input: per-network edge attribute tables ───────────────────────────────────
edge_df_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Research/Global road network resilience/01_data/edge_attri_df'

# ── Accumulators ──────────────────────────────────────────────────────────────
netids = []
ginis = []

# ── Compute Gini coefficient for each network ─────────────────────────────────
# Loop through the files
for file in tqdm(os.listdir(edge_df_dir)):
    if file.endswith("csv"):
        net_df = pd.read_csv(os.path.join(edge_df_dir,file), index_col=0)
        # Extract network ID from filename token
        net_id = int(file.split('_')[4]) # extract network id

        # Calculate the variance of edge frequency
        edge_gini = gini(net_df.dry_route_freq)

        # Append to lists
        netids.append(net_id)
        ginis.append(edge_gini)

# ── Assemble and export results ───────────────────────────────────────────────
edge_gini_df = pd.DataFrame({
    'net_id': netids,
    'edge_freq_gini': ginis
})

# Save to csv file
edge_gini_df.to_csv('/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Research/Global road network resilience/01_data/csv/edge_freq_gini.csv')

100%|██████████| 2615/2615 [00:08<00:00, 295.19it/s]


---
## Section 10: Spatial Homogeneity of Road Distribution

### Motivation
The Gini and variance metrics above capture *topological* inequality in routing load across edges. This section measures a complementary property: **spatial homogeneity** — how evenly road infrastructure is distributed across the physical space of the urban area.

A spatially homogeneous network has road coverage spread uniformly across the city; a heterogeneous network has dense coverage in some zones and sparse or absent coverage in others. Spatial homogeneity is relevant to equity of access and to the spatial patterns of flood vulnerability.

### Methodology
1. **Grid construction**: The convex hull of the network's nodes is partitioned into a regular grid of 200 m × 200 m cells. Only cells that intersect the convex hull are retained.
2. **Road length per cell**: For each grid cell, the total clipped length of road edges intersecting that cell is computed. Edges without explicit geometry attributes are reconstructed as straight-line `LineString` segments between their endpoint node coordinates.
3. **Homogeneity metrics** (computed on the per-cell road length distribution):

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **Shannon evenness** ($E$) | $E = H / \ln(m)$, where $H = -\sum p_i \ln p_i$ | 1 = perfectly uniform; 0 = all roads in one cell |
| **Gini coefficient** ($G$) | See Section 9 | 0 = uniform; 1 = maximally concentrated |

### CRS considerations
- Node coordinates from OSMnx are in **EPSG:4326** (geographic).
- All geometric operations (grid construction, edge intersections, length computation) are performed in **EPSG:3857** (Web Mercator), a projected CRS using meters.
- Lengths computed in EPSG:3857 are approximate; for high-precision work, an equal-area projection would be preferred.

### Performance note
This section is computationally intensive (~523 s/network × 2,616 networks ≈ 380 CPU-hours). The bottleneck is the per-cell edge intersection loop. Vectorised alternatives (e.g., spatial joins with `geopandas.overlay`) would substantially reduce runtime.

In [ ]:
# ── Input ─────────────────────────────────────────────────────────────────────
graph_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology(2)/Research/Global road network resilience/01_data/all_graph_flooded'

# ── Accumulators ──────────────────────────────────────────────────────────────
netids = []
ginis = []
evens = []

# ── Spatial homogeneity computation ───────────────────────────────────────────
for file in tqdm(os.listdir(graph_dir)):
    if file.endswith('pk'):

        # Extract network id
        net_id = int(re.search(r'_r_(\d+)_', file).group(1))
        
        # Load graph file
        g = pickle.load(open(os.path.join(graph_dir, file), 'rb'))

        # extract node coordinates
        nodes = [Point(data['x'], data['y']) for n, data in g.nodes(data=True)]
        nodes_gdf = gpd.GeoDataFrame(geometry=nodes, crs="EPSG:4326")  # make sure projected; EPSG:3857 is WGS 84 / Pseudo-Mercator
        nodes_gdf = nodes_gdf.to_crs("EPSG:3857")


        # Compute convex hull
        convex_hull = nodes_gdf.unary_union.convex_hull

        # Create grid within convex hull
        cell_size = 200  # meters
        minx, miny, maxx, maxy = convex_hull.bounds
        cols = int(np.ceil((maxx - minx) / cell_size))
        rows = int(np.ceil((maxy - miny) / cell_size))

        # Only retain cells that intersect the convex hull (excludes corners outside)
        grid_cells = []
        for i in range(cols):
            for j in range(rows):
                cell = box(minx + i*cell_size, miny + j*cell_size,
                        minx + (i+1)*cell_size, miny + (j+1)*cell_size)
                # only keep cells that intersect the convex hull
                if cell.intersects(convex_hull):
                    grid_cells.append(cell)

        grid = gpd.GeoDataFrame({'geometry': grid_cells}, crs="EPSG:3857")

        # Compute Road Length per Grid Cell
        # Convert edges to GeoDataFrame
        edges = []
        for u, v, data in g.edges(data=True):
            if 'geometry' in data:
                edges.append(data['geometry'])
            else:
                x1, y1 = g.nodes[u]['x'], g.nodes[u]['y']
                x2, y2 = g.nodes[v]['x'], g.nodes[v]['y']
                edges.append(LineString([(x1, y1), (x2, y2)]))

        edges_gdf = gpd.GeoDataFrame(geometry=edges, crs="EPSG:4326")
        edges_gdf = edges_gdf.to_crs("EPSG:3857")

        # Compute road length per cell
        grid['road_length'] = 0.0
        for idx, cell in grid.iterrows():
            roads_in_cell = edges_gdf[edges_gdf.intersects(cell['geometry'])]
            total_length = roads_in_cell.geometry.intersection(cell['geometry']).length.sum()
            grid.at[idx, 'road_length'] = total_length

        # Compute Shannon evenness
        p = grid['road_length'] / grid['road_length'].sum()
        p = p[p > 0]  # ignore empty cells

        H = -np.sum(p * np.log(p))
        E = H / np.log(len(p))

        # Compute Gini coefficient
        def gini(array):
            array = np.array(array).flatten()
            array = array - np.min(array) + 1e-9  # shift to positive
            array = np.sort(array)
            n = array.size
            index = np.arange(1, n+1)
            return ((np.sum((2*index - n - 1)*array)) / (n*np.sum(array)))

        G = gini(grid['road_length'])

        netids.append(net_id)
        ginis.append(G)
        evens.append(E)

# Create new DataFrame
edge_homogeneity_df = pd.DataFrame({
    'net_id': netids,
    'net_gini': ginis,
    'net_shannon': evens
})

# Save to csv file
edge_homogeneity_df.to_csv('/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology(2)/Research/Global road network resilience/01_data/csv/road_homogeneity_200m.csv')